In [ ]:
!pip install mlflow==2.19.0 dagshub scikit-learn matplotlib seaborn pandas numpy

In [ ]:
!dagshub login

                      ❗❗❗ AUTHORIZATION REQUIRED ❗❗❗                      


Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=be5acee9-4e9b-4257-9d6e-ebbe508049f9&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=848c236506b926c2c367de04e69c394b1b892e1158ef0612b79dfa7dada0b5d0


⠸ Waiting for authorization
✅ OAuth token added


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def train_with_tuning():
    # 1. INISIALISASI DAGSHUB (Wajib untuk Online Tracking - Poin Advance)
    # Ganti dengan username dan nama repo DagsHub milikmu sendiri!
    dagshub.init(repo_owner='BungaRasikhahhaya', repo_name='Eksperimen_SML_BungaRasikhahHaya', mlflow=True)
    mlflow.set_experiment("Eksperimen_Model_Tuning_Advance")

    # 2. Load Data Preprocessing dari folder yang sudah disiapkan
    data_dir = "Membangun_model/heart_disease_preprocessing" # Sesuaikan nama folder datasetmu
    X_train = pd.read_csv(f"{data_dir}/X_train.csv")
    X_test = pd.read_csv(f"{data_dir}/X_test.csv")
    y_train = pd.read_csv(f"{data_dir}/y_train.csv").values.ravel()
    y_test = pd.read_csv(f"{data_dir}/y_test.csv").values.ravel()

    # 3. Proses Hyperparameter Tuning menggunakan GridSearchCV
    print("⏳ Sedang mencari kombinasi hyperparameter terbaik...")
    base_model = RandomForestClassifier(random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 150],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }

    grid_search = GridSearchCV(estimator=base_model, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_

    # 4. Manual Logging ke MLflow Tracking UI (Syarat mutlak Skilled/Advance)
    with mlflow.start_run(run_name="Random_Forest_Tuning_Advance"):

        # Log parameter terbaik hasil tuning secara manual
        mlflow.log_params(grid_search.best_params_)

        # Evaluasi performa model
        y_pred = best_model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        # Log metrik akurasi secara manual
        mlflow.log_metric("accuracy", acc)

        # Simpan model utama ke MLflow
        mlflow.sklearn.log_model(best_model, "model_rf_tuned")

        # --- BONUS ADVANCE: BUAT DAN LOG MINIMAL 2 ARTEFAK TAMBAHAN ---
        os.makedirs("artifacts", exist_ok=True)

        # Artefak 1: File teks laporan klasifikasi
        report_path = "artifacts/classification_report.txt"
        with open(report_path, "w") as f:
            f.write(classification_report(y_test, y_pred))
        mlflow.log_artifact(report_path)

        # Artefak 2: Gambar visualisasi Confusion Matrix
        plt.figure(figsize=(6,5))
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title('Confusion Matrix')
        plt.ylabel('Actual Label')
        plt.xlabel('Predicted Label')
        plot_path = "artifacts/confusion_matrix.png"
        plt.savefig(plot_path)
        plt.close()
        mlflow.log_artifact(plot_path)

        print(f"\n🎯 Proses Selesai! Model terbaik berhasil dilatih.")
        print(f"Akurasi Akhir: {acc:.4f}")
        print("✅ Semua metrik, parameter, dan artefak berhasil dikirim ke DagsHub!")

# Jalankan fungsi training
train_with_tuning()

Initialized MLflow to track repo "BungaRasikhahhaya/Eksperimen_SML_BungaRasikhahHaya"

Repository BungaRasikhahhaya/Eksperimen_SML_BungaRasikhahHaya initialized!

⏳ Sedang mencari kombinasi hyperparameter terbaik...


2026/05/29 15:56:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



🎯 Proses Selesai! Model terbaik berhasil dilatih.
Akurasi Akhir: 0.8525
✅ Semua metrik, parameter, dan artefak berhasil dikirim ke DagsHub!
🏃 View run Random_Forest_Tuning_Advance at: https://dagshub.com/BungaRasikhahhaya/Eksperimen_SML_BungaRasikhahHaya.mlflow/#/experiments/0/runs/2e2924d1d81b44429c2b1a223166e548
🧪 View experiment at: https://dagshub.com/BungaRasikhahhaya/Eksperimen_SML_BungaRasikhahHaya.mlflow/#/experiments/0
